# Лабораторная работа № 7 — Функции и процедуры

Вариант 11 «Автостоянка». Задание на стр. 124–130 хранится в [tasks/lab7.md](../../tasks/lab7.md).

Сначала ЛР2 → ЛР3, затем **Run All**. ЛР6 для запуска не требуется.
Подключение через переменные PG*, по умолчанию `parking` на `127.0.0.1:5432`.
Определения индивидуальных подпрограмм сохраняются в `lab7`, учебных — в `lab7_demo`.
Изменения тестовых данных отменяются savepoint; COMMIT в конце сохраняет определения.
Повторный запуск поддерживается при неизменных сигнатурах подпрограмм.
При прерывании выполните `conn.rollback()` и повторите с начала.

[Отчёт](ОТЧЁТ_ЛР7_Автостоянка.md) · [Все лабораторные](../../README.md)


In [1]:
import os
import getpass
import pandas as pd
from IPython.display import display
try:
    import psycopg
except ImportError:
    import psycopg2 as psycopg

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)
# При необходимости задайте PGHOST, PGPORT, PGUSER, PGPASSWORD перед запуском.
# Пароль в ноутбуке не сохраняйте: драйвер также поддерживает ~/.pgpass.
DB_NAME = os.getenv("PGDATABASE", "parking")
conn = psycopg.connect(
    host=os.getenv("PGHOST", "127.0.0.1"),
    port=int(os.getenv("PGPORT", "5432")),
    user=os.getenv("PGUSER", getpass.getuser()),
    password=os.getenv("PGPASSWORD") or None,
    dbname=DB_NAME,
)
conn.autocommit = False

def q(sql, params=None):
    """Выполнить инструкцию; выборку показать таблицей."""
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            if cur.description:
                frame = pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
                frame.index = range(1, len(frame) + 1)
                display(frame)
                print(f"Строк: {len(frame)}")
            else:
                print(cur.statusmessage)
    except Exception:
        conn.rollback()
        raise

run = q
print("Подключение:", DB_NAME)


Подключение: parking_lab7_final


## Подготовка схем

Учебные примеры отделены от трёх процедур и двух функций индивидуального задания.

In [2]:
q("""
CREATE SCHEMA IF NOT EXISTS lab7;
CREATE SCHEMA IF NOT EXISTS lab7_demo;
""")

CREATE SCHEMA


## Учебные примеры со скалярным результатом

Адаптация стр. 124–125: присваивание результата SELECT через INTO и компактный RETURN. На стр. 127 приведена функция SQL со скалярным результатом.

In [3]:
q("""
CREATE OR REPLACE FUNCTION lab7_demo.client_phone_into(p_client_id integer)
RETURNS text LANGUAGE plpgsql STABLE AS $$
DECLARE result text;
BEGIN
    SELECT cl_phone INTO result
    FROM public.clients WHERE cl_id = p_client_id;
    RETURN result;
END;
$$;
""")

CREATE FUNCTION


In [4]:
q("""
CREATE OR REPLACE FUNCTION lab7_demo.client_phone_direct(p_client_id integer)
RETURNS text LANGUAGE plpgsql STABLE AS $$
BEGIN
    RETURN (SELECT cl_phone FROM public.clients WHERE cl_id = p_client_id);
END;
$$;
""")

CREATE FUNCTION


In [5]:
q("""
CREATE OR REPLACE FUNCTION lab7_demo.hello(p text)
RETURNS text LANGUAGE sql IMMUTABLE AS $$
    SELECT 'Hello, ' || p || '!';
$$;
""")

CREATE FUNCTION


In [6]:
q("""
SELECT lab7_demo.client_phone_into(1) AS with_into,
       lab7_demo.client_phone_direct(1) AS direct_return,
       lab7_demo.hello('parking') AS greeting;
""")

,with_into,direct_return,greeting
1,+79139990011,+79139990011,"Hello, parking!"


Строк: 1


## Учебные примеры с составным результатом

Адаптация стр. 125–128: составной тип, FOR с RETURN NEXT, SQL-функция с SETOF и SQL-функция без SETOF. Автомобили заменяют товары пособия. Без SETOF возвращается только первая строка; для прикладной выборки следует явно задавать требуемую кардинальность.

In [7]:
q("""
DO $$
BEGIN
    IF NOT EXISTS (
        SELECT 1 FROM pg_type t JOIN pg_namespace s ON s.oid = t.typnamespace
        WHERE s.nspname = 'lab7_demo' AND t.typname = 'vehicle_info'
    ) THEN
        CREATE TYPE lab7_demo.vehicle_info AS (code integer, plate text);
    END IF;
END;
$$;
""")

DO


In [8]:
q("""
CREATE OR REPLACE FUNCTION lab7_demo.vehicle_loop()
RETURNS SETOF lab7_demo.vehicle_info LANGUAGE plpgsql STABLE AS $$
DECLARE v record;
BEGIN
    FOR v IN SELECT veh_id, veh_plate FROM public.vehicles ORDER BY veh_id
    LOOP
        RETURN NEXT ROW(v.veh_id, v.veh_plate)::lab7_demo.vehicle_info;
    END LOOP;
    RETURN;
END;
$$;
""")

CREATE FUNCTION


In [9]:
q("""
CREATE OR REPLACE FUNCTION lab7_demo.vehicle_sql()
RETURNS SETOF lab7_demo.vehicle_info LANGUAGE sql STABLE AS $$
    SELECT ROW(veh_id, veh_plate)::lab7_demo.vehicle_info
    FROM public.vehicles ORDER BY veh_id;
$$;
""")

CREATE FUNCTION


In [10]:
q("""
CREATE OR REPLACE FUNCTION lab7_demo.vehicle_first()
RETURNS lab7_demo.vehicle_info LANGUAGE sql STABLE AS $$
    SELECT ROW(veh_id, veh_plate)::lab7_demo.vehicle_info
    FROM public.vehicles ORDER BY veh_id;
$$;
""")

CREATE FUNCTION


In [11]:
q("""
SELECT * FROM lab7_demo.vehicle_loop() ORDER BY code LIMIT 5;
""")

,code,plate
1,1,А123ВС55
2,2,В456ЕК55
3,3,Е789МН55
4,4,К321ОР55
5,5,М654СТ55


Строк: 5


In [12]:
q("""
SELECT * FROM lab7_demo.vehicle_sql() ORDER BY code LIMIT 4;
""")

,code,plate
1,1,А123ВС55
2,2,В456ЕК55
3,3,Е789МН55
4,4,К321ОР55


Строк: 4


In [13]:
q("""
SELECT * FROM lab7_demo.vehicle_first();
""")

,code,plate
1,1,А123ВС55


Строк: 1


## Процедура добавления клиента

Адаптация процедуры вставки на стр. 129. Добавляется физическое лицо с нулевой скидкой. Ключ передаётся явно, как в исходной схеме. Ограничения таблицы проверяют уникальность документа и допустимость полей. Процедуры вызываются через CALL и не выполняют собственный COMMIT.

In [14]:
q("""
CREATE OR REPLACE PROCEDURE lab7.client_insert(
    p_id integer, p_name text, p_doc text, p_phone text,
    p_reg_date date DEFAULT CURRENT_DATE
)
LANGUAGE plpgsql AS $$
BEGIN
    INSERT INTO public.clients (
        cl_id, cl_type, cl_name, cl_doc, cl_phone,
        cl_email, cl_discount, cl_reg_date
    ) VALUES (
        p_id, 'Ф', p_name, p_doc, p_phone, NULL, 0, p_reg_date
    );
END;
$$;
""")

CREATE PROCEDURE


## Процедура изменения контактов и скидки

Адаптация процедуры обновления на стр. 129. При отсутствии клиента возбуждается P0002. Диапазон скидки 0–30 проверяет существующее ограничение таблицы; неверное значение вызывает 23514.

In [15]:
q("""
CREATE OR REPLACE PROCEDURE lab7.client_update(
    p_id integer, p_phone text, p_email text, p_discount integer
)
LANGUAGE plpgsql AS $$
BEGIN
    UPDATE public.clients
    SET cl_phone = p_phone, cl_email = p_email, cl_discount = p_discount
    WHERE cl_id = p_id;
    IF NOT FOUND THEN
        RAISE EXCEPTION 'Клиент % не найден', p_id USING ERRCODE = 'P0002';
    END IF;
END;
$$;
""")

CREATE PROCEDURE


## Процедура удаления клиента без связанных данных

Адаптация процедуры удаления на стр. 130. Удаление допускается для клиента без автомобилей и договоров. Для связанных клиентов процедура выдаёт 23503, чтобы не запускать каскадное удаление финансовой истории. FOR UPDATE блокирует выбранного клиента до конца транзакции. Правила внешних ключей исходной схемы не изменяются.

In [16]:
q("""
CREATE OR REPLACE PROCEDURE lab7.client_delete(p_id integer)
LANGUAGE plpgsql AS $$
DECLARE locked_id integer;
BEGIN
    SELECT cl_id INTO locked_id FROM public.clients
    WHERE cl_id = p_id FOR UPDATE;
    IF NOT FOUND THEN
        RAISE EXCEPTION 'Клиент % не найден', p_id USING ERRCODE = 'P0002';
    END IF;
    IF EXISTS (SELECT 1 FROM public.vehicles WHERE veh_client_id = p_id)
       OR EXISTS (SELECT 1 FROM public.contracts WHERE contr_client_id = p_id)
    THEN
        RAISE EXCEPTION 'У клиента % есть связанные данные', p_id
            USING ERRCODE = '23503';
    END IF;
    DELETE FROM public.clients WHERE cl_id = p_id;
END;
$$;
""")

CREATE PROCEDURE


## Проверка трёх процедур

В savepoint создаётся отдельный учебный клиент, затем меняются его контакты и скидка, после чего он удаляется. Тестовый код выбирается как MAX(cl_id)+100 в изолированном наборе данных; это демонстрационный приём, а не генератор ключей для конкурентной работы. Уникальный документ генерируется отдельно. После демонстрации savepoint отменяется.

In [17]:
import uuid
with conn.cursor() as cur:
    cur.execute("SELECT COALESCE(MAX(cl_id), 0) + 100 FROM public.clients")
    demo_id = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM public.clients")
    original_clients = cur.fetchone()[0]
q("SAVEPOINT client_demo")
q("CALL lab7.client_insert(%s, %s, %s, %s, DATE '2024-10-31')",
  (demo_id, 'Учебный клиент ЛР7', 'ЛР7-' + uuid.uuid4().hex[:16], '+70000000000'))
q("SELECT cl_id, cl_name, cl_phone, cl_discount FROM public.clients WHERE cl_id=%s", (demo_id,))

SAVEPOINT
CALL


,cl_id,cl_name,cl_phone,cl_discount
1,108,Учебный клиент ЛР7,+70000000000,0


Строк: 1


In [18]:
q("CALL lab7.client_update(%s, %s, %s, %s)",
  (demo_id, '+70000000001', 'lab7@example.test', 10))
q("SELECT cl_id, cl_phone, cl_email, cl_discount FROM public.clients WHERE cl_id=%s", (demo_id,))
with conn.cursor() as cur:
    cur.execute("SELECT cl_discount FROM public.clients WHERE cl_id=%s", (demo_id,))
    assert cur.fetchone()[0] == 10

CALL


,cl_id,cl_phone,cl_email,cl_discount
1,108,+70000000001,lab7@example.test,10


Строк: 1


In [19]:
q("CALL lab7.client_delete(%s)", (demo_id,))
q("SELECT COUNT(*) AS demo_clients_after_delete FROM public.clients WHERE cl_id=%s", (demo_id,))
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM public.clients WHERE cl_id=%s", (demo_id,))
    assert cur.fetchone()[0] == 0
q("ROLLBACK TO SAVEPOINT client_demo")
q("RELEASE SAVEPOINT client_demo")
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM public.clients")
    assert cur.fetchone()[0] == original_clients
print("Вставка, обновление и удаление проверены; число клиентов восстановлено.")

CALL


,demo_clients_after_delete
1,0


Строк: 1
ROLLBACK
RELEASE
Вставка, обновление и удаление проверены; число клиентов восстановлено.


## Функция расчёта остатка по договору

Функция PL/pgSQL возвращает сумму договора минус все его платежи. SELECT INTO STRICT требует существования одного договора. Отрицательный остаток означает переплату; для отсутствующего договора выдаётся P0002. STABLE указывает, что функция читает данные и не меняет их.

In [20]:
q("""
CREATE OR REPLACE FUNCTION lab7.contract_balance(p_contract_id integer)
RETURNS numeric LANGUAGE plpgsql STABLE AS $$
DECLARE
    amount numeric;
    paid numeric;
BEGIN
    SELECT contr_amount INTO STRICT amount
    FROM public.contracts WHERE contr_id = p_contract_id;
    SELECT COALESCE(SUM(pay_sum), 0) INTO paid
    FROM public.payments WHERE pay_contr_id = p_contract_id;
    RETURN amount - paid;
END;
$$;
""")

CREATE FUNCTION


In [21]:
q("""
SELECT contr_number, contr_amount,
       lab7.contract_balance(contr_id) AS balance
FROM public.contracts ORDER BY contr_id;
""")

,contr_number,contr_amount,balance
1,Д-2024/001,25650.00,0.00
2,Д-2024/002,27000.00,0.00
3,Д-2024/003,97200.00,0.00
4,Д-2024/004,33000.00,0.00
5,Д-2024/005,12825.00,0.00
6,Д-2024/006,91800.00,45900.00
7,Д-2024/007,15000.00,0.00
8,Д-2024/008,16500.00,11000.00
9,Д-2024/009,5225.00,-5225.00
10,Д-2024/010,3500.00,0.00


Строк: 10


## Функция выборки договоров клиента

SQL-функция возвращает таблицу договоров указанного клиента. Вызов находится в FROM; для клиента без договоров возвращается пустое множество. Сортировка задаётся во внешнем запросе.

In [22]:
q("""
CREATE OR REPLACE FUNCTION lab7.client_contracts(p_client_id integer)
RETURNS TABLE (
    contract_id integer, contract_number text,
    amount numeric, status text
)
LANGUAGE sql STABLE AS $$
    SELECT contr_id, contr_number::text, contr_amount, contr_status
    FROM public.contracts WHERE contr_client_id = p_client_id;
$$;
""")

CREATE FUNCTION


In [23]:
q("""
SELECT * FROM lab7.client_contracts(1) ORDER BY contract_id;
""")

,contract_id,contract_number,amount,status
1,1,Д-2024/001,25650.00,действующий
2,9,Д-2024/009,5225.00,действующий


Строк: 2


## Проверка ошибок и сохранности данных

Проверяются: неизвестный клиент, скидка вне диапазона, дублирование документа, попытка удалить связанного клиента и запрос остатка неизвестного договора. Каждая ожидаемая ошибка изолирована savepoint. Проверка сравнивает содержимое всех девяти исходных таблиц до и после тестов.

In [24]:
def expect_error(sql, params, expected_state):
    with conn.cursor() as cur:
        cur.execute("SAVEPOINT expected_error")
        try:
            cur.execute(sql, params)
        except psycopg.Error as exc:
            state = getattr(exc, "sqlstate", None) or getattr(exc, "pgcode", None)
            cur.execute("ROLLBACK TO SAVEPOINT expected_error")
            if state != expected_state:
                raise AssertionError((state, expected_state)) from exc
            print(f"Ожидаемая ошибка {state}: {sql.split('(')[0]}")
        else:
            cur.execute("ROLLBACK TO SAVEPOINT expected_error")
            raise AssertionError("Ожидаемая ошибка не возникла")
        finally:
            cur.execute("RELEASE SAVEPOINT expected_error")

tables = ["vehicle_categories", "clients", "employees", "vehicles", "places",
          "tariffs", "contracts", "payments", "passages"]
def snapshot():
    with conn.cursor() as cur:
        result = {}
        for table in tables:
            # Имена взяты только из фиксированного списка выше.
            cur.execute(f"SELECT row_to_json(t)::text FROM public.{table} t ORDER BY row_to_json(t)::text")
            result[table] = cur.fetchall()
        return result
before_checks = snapshot()
with conn.cursor() as cur:
    cur.execute("SELECT cl_id, cl_doc, cl_phone FROM public.clients ORDER BY cl_id LIMIT 1")
    client_id, existing_doc, phone = cur.fetchone()
    cur.execute("SELECT contr_client_id FROM public.contracts ORDER BY contr_id LIMIT 1")
    linked_client = cur.fetchone()[0]
    cur.execute("SELECT COALESCE(MAX(contr_id), 0) + 100 FROM public.contracts")
    missing_contract = cur.fetchone()[0]
expect_error("CALL lab7.client_update(%s, %s, %s, %s)",
             (demo_id, phone, None, 0), "P0002")
expect_error("CALL lab7.client_update(%s, %s, %s, %s)",
             (client_id, phone, None, 31), "23514")
expect_error("CALL lab7.client_insert(%s, %s, %s, %s)",
             (demo_id, 'Дубликат документа', existing_doc, phone), "23505")
expect_error("CALL lab7.client_delete(%s)", (linked_client,), "23503")
expect_error("CALL lab7.client_delete(%s)", (demo_id,), "P0002")
expect_error("SELECT lab7.contract_balance(%s)", (missing_contract,), "P0002")
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM lab7.client_contracts(%s)", (demo_id,))
    assert cur.fetchone()[0] == 0
    cur.execute("SELECT c.contr_id, c.contr_amount - COALESCE(SUM(p.pay_sum), 0), lab7.contract_balance(c.contr_id) FROM public.contracts c LEFT JOIN public.payments p ON p.pay_contr_id=c.contr_id GROUP BY c.contr_id ORDER BY c.contr_id")
    assert all(expected == actual for _, expected, actual in cur.fetchall())
assert snapshot() == before_checks
print("Все проверки пройдены: ошибки обработаны; остатки совпали с прямым запросом.")
print("Содержимое девяти исходных таблиц после тестов не изменилось.")

Ожидаемая ошибка P0002: CALL lab7.client_update
Ожидаемая ошибка 23514: CALL lab7.client_update
Ожидаемая ошибка 23505: CALL lab7.client_insert
Ожидаемая ошибка 23503: CALL lab7.client_delete
Ожидаемая ошибка P0002: CALL lab7.client_delete
Ожидаемая ошибка P0002: SELECT lab7.contract_balance
Все проверки пройдены: ошибки обработаны; остатки совпали с прямым запросом.
Содержимое девяти исходных таблиц после тестов не изменилось.


## Состав сохранённых подпрограмм

В индивидуальной схеме lab7 должны находиться ровно три процедуры и две функции. Шесть учебных функций и составной тип находятся отдельно в lab7_demo.

In [25]:
q("""
SELECT p.proname AS routine_name,
       CASE p.prokind WHEN 'p' THEN 'PROCEDURE' ELSE 'FUNCTION' END AS kind,
       l.lanname AS language
FROM pg_proc p
JOIN pg_namespace s ON s.oid = p.pronamespace
JOIN pg_language l ON l.oid = p.prolang
WHERE s.nspname = 'lab7'
ORDER BY kind DESC, routine_name;
""")

,routine_name,kind,language
1,client_delete,PROCEDURE,plpgsql
2,client_insert,PROCEDURE,plpgsql
3,client_update,PROCEDURE,plpgsql
4,client_contracts,FUNCTION,sql
5,contract_balance,FUNCTION,plpgsql


Строк: 5


In [26]:
with conn.cursor() as cur:
    cur.execute("SELECT prokind, COUNT(*) FROM pg_proc p JOIN pg_namespace s ON s.oid=p.pronamespace WHERE s.nspname='lab7' GROUP BY prokind")
    assert dict(cur.fetchall()) == {'p': 3, 'f': 2}
conn.commit()
conn.close()
print("ЛР7 выполнена: три процедуры и две функции сохранены; учебные изменения отменены.")

ЛР7 выполнена: три процедуры и две функции сохранены; учебные изменения отменены.
